In [4]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh

In [5]:
def euclidean(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


def eye_aspect_ratio(landmarks, eye_indices, w, h):
    pts = [(int(landmarks[i].x * w), int(landmarks[i].y * h)) for i in eye_indices]

    p1, p2, p3, p4, p5, p6 = pts

    vertical_1 = euclidean(p2, p6)
    vertical_2 = euclidean(p3, p5)
    horizontal = euclidean(p1, p4)

    if horizontal == 0:
        return 0

    return (vertical_1 + vertical_2) / (2.0 * horizontal)


LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

# For tuning
blink_threshold = 0.21

In [ ]:
# Testing blink detection

cap = cv2.VideoCapture(0)

blink_count = 0
eye_closed = False

window_name = "Blink Detection Test"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 1000, 700)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark

            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            cv2.putText(frame, f"Blinks: {blink_count}", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            if avg_ear < blink_threshold and not eye_closed:
                eye_closed = True

            if avg_ear >= blink_threshold and eye_closed:
                blink_count += 1
                eye_closed = False

        cv2.imshow(window_name, frame)
        # press 'q' to quit
        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        # or quit if window is closed normally
        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776740538.386700   19395 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776740538.436413   44544 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776740538.437408   44533 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776740538.448819   44532 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [14]:
def get_point(landmarks, idx, w, h):
    return np.array([
        int(landmarks[idx].x * w),
        int(landmarks[idx].y * h)
    ])

# For tuning
turn_threshold = 0.07

In [ ]:
# Testing head turn detection

cap = cv2.VideoCapture(0)

window_name = "Head Turn Detection Test"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 1000, 700)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        direction = "No face"

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark

            nose = get_point(face_landmarks, 1, w, h)
            left_face = get_point(face_landmarks, 234, w, h)
            right_face = get_point(face_landmarks, 454, w, h)

            face_center_x = (left_face[0] + right_face[0]) / 2
            nose_offset = (nose[0] - face_center_x) / w

            if nose_offset < -turn_threshold:
                direction = "Turned Left"
            elif nose_offset > turn_threshold:
                direction = "Turned Right"
            else:
                direction = "Forward"

            # draw landmarks used
            cv2.circle(frame, tuple(nose), 4, (0, 0, 255), -1)
            cv2.circle(frame, tuple(left_face), 4, (255, 0, 0), -1)
            cv2.circle(frame, tuple(right_face), 4, (255, 0, 0), -1)

            cv2.putText(frame, f"Offset: {nose_offset:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.putText(frame, f"Direction: {direction}", (30, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        cv2.imshow(window_name, frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776774734.728735  402104 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776774734.793491  406336 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776774734.796254  406324 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776774734.809284  406323 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [14]:
# Combination of head turn and blink detection

cap = cv2.VideoCapture(0)

window_name = "Fixed Liveness Challenge"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 1000, 700)

blink_count = 0
eye_closed = False
challenge_step = 1
liveness_passed = False

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        avg_ear = 0.0
        direction = "No face"

        if results.multi_face_landmarks:
            face_landmarks = results.multi_face_landmarks[0].landmark


            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            # ----- Head turn detection -----
            nose = get_point(face_landmarks, 1, w, h)
            left_face = get_point(face_landmarks, 234, w, h)
            right_face = get_point(face_landmarks, 454, w, h)

            face_center_x = (left_face[0] + right_face[0]) / 2
            nose_offset = (nose[0] - face_center_x) / w

            if nose_offset < -turn_threshold:
                direction = "Turned Left"
            elif nose_offset > turn_threshold:
                direction = "Turned Right"
            else:
                direction = "Forward"

            # Draw debug points
            cv2.circle(frame, tuple(nose), 4, (0, 0, 255), -1)
            cv2.circle(frame, tuple(left_face), 4, (255, 0, 0), -1)
            cv2.circle(frame, tuple(right_face), 4, (255, 0, 0), -1)

            if challenge_step == 1:
                instruction = "Step 1: Turn head left"

                if direction == "Turned Left":
                    challenge_step = 2
                    blink_count = 0
                    eye_closed = False

            elif challenge_step == 2:
                instruction = "Step 2: Blink once"

                if avg_ear < blink_threshold and not eye_closed:
                    eye_closed = True

                if avg_ear >= blink_threshold and eye_closed:
                    blink_count += 1
                    eye_closed = False

                if blink_count >= 1:
                    challenge_step = 3
                    liveness_passed = True

            else:
                instruction = "Liveness Passed"

            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            cv2.putText(frame, f"Blinks: {blink_count}", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
            cv2.putText(frame, f"Direction: {direction}", (30, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 255), 2)
            cv2.putText(frame, instruction, (30, 170),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

            if liveness_passed:
                cv2.putText(frame, "Verified as live user", (30, 220),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)
        else:
            cv2.putText(frame, "No face detected", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

        cv2.imshow(window_name, frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776741976.958238   19395 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776741977.004280   64827 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776741977.005890   64814 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776741977.016287   64817 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [15]:
# Added randomized challenge with delays between steps
# Challenge: Random turn -> Blink random times -> Random turn -> Hold still (2 seconds) -> Blink once
# Restart challenge if more than 1 face detected, or face goes out of frame
# Press 'r' to try another challenge
import random
import time


def make_random_challenge():
    first_turn = random.choice(["turn_left", "turn_right"])
    second_turn = random.choice(["turn_left", "turn_right"])
    blink_target = random.randint(2, 3)

    return [
        {"type": first_turn},
        {"type": "blink", "count": blink_target},
        {"type": second_turn},
        {"type": "forward_hold", "duration": 2.0},
    ]


def step_to_instruction(step):
    if step["type"] == "turn_left":
        return "Turn head left"
    elif step["type"] == "turn_right":
        return "Turn head right"
    elif step["type"] == "forward":
        return "Face forward"
    elif step["type"] == "blink":
        count = step["count"]
        if count == 1:
            return "Face forward and blink once"
        return f"Face forward and blink {count} times"
    elif step["type"] == "forward_hold":
        return f"Face forward and hold still for {step['duration']:.0f} seconds"
    return ""


def make_new_state(step_delay_seconds):
    return {
        "challenge": make_random_challenge(),
        "step": 0,
        "passed": False,
        "waiting_until": time.time() + step_delay_seconds,
        "blink_count": 0,
        "closed_frames": 0,
        "hold_started_at": None,
        "awaiting_final_blink": False,
        "post_capture_blink_count": 0,
        "post_capture_closed_frames": 0,
        "captured_face_frame": None,
    }


cap = cv2.VideoCapture(0)

window_name = "Randomized Liveness Challenge"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.resizeWindow(window_name, 1000, 700)

forward_tolerance = 0.03
min_closed_frames = 2
step_delay_seconds = 1.5
face_retry_delay_seconds = 2.0

state = make_new_state(step_delay_seconds)
current_instruction = None
retry_available_at = None
print("Selected challenge:", [step_to_instruction(s) for s in state["challenge"]])

with mp_face_mesh.FaceMesh(
    max_num_faces=2,  # To detect multiple faces
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as face_mesh:

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)

        avg_ear = 0.0
        direction = "No face"
        nose_offset = 0.0
        now = time.time()

        if retry_available_at is not None and now >= retry_available_at:
            retry_available_at = None
            current_instruction = None

        if results.multi_face_landmarks and len(results.multi_face_landmarks) >= 2:
            if retry_available_at is None:
                state = make_new_state(step_delay_seconds)
                retry_available_at = now + face_retry_delay_seconds
                current_instruction = None

            remaining = max(0.0, retry_available_at - now) if retry_available_at is not None else face_retry_delay_seconds
            cv2.putText(frame, "Keep only one face within the frame", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.putText(frame, f"Retrying in {remaining:.1f}s", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

        elif not results.multi_face_landmarks:
            if retry_available_at is None:
                state = make_new_state(step_delay_seconds)
                retry_available_at = now + face_retry_delay_seconds
                current_instruction = None

            remaining = max(0.0, retry_available_at - now) if retry_available_at is not None else face_retry_delay_seconds
            cv2.putText(frame, "Keep your face within the frame", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.putText(frame, f"Retrying in {remaining:.1f}s", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)

        else:
            face_landmarks = results.multi_face_landmarks[0].landmark

            left_ear = eye_aspect_ratio(face_landmarks, LEFT_EYE, w, h)
            right_ear = eye_aspect_ratio(face_landmarks, RIGHT_EYE, w, h)
            avg_ear = (left_ear + right_ear) / 2.0

            nose = get_point(face_landmarks, 1, w, h)
            left_face = get_point(face_landmarks, 234, w, h)
            right_face = get_point(face_landmarks, 454, w, h)

            face_center_x = (left_face[0] + right_face[0]) / 2
            nose_offset = (nose[0] - face_center_x) / w

            if nose_offset < -turn_threshold:
                direction = "Turned Left"
            elif nose_offset > turn_threshold:
                direction = "Turned Right"
            elif abs(nose_offset) <= forward_tolerance:
                direction = "Forward"
            else:
                direction = "Slight Turn"

            cv2.circle(frame, tuple(nose), 4, (0, 0, 255), -1)
            cv2.circle(frame, tuple(left_face), 4, (255, 0, 0), -1)
            cv2.circle(frame, tuple(right_face), 4, (255, 0, 0), -1)

            if not state["passed"] and not state["awaiting_final_blink"]:
                if now < state["waiting_until"]:
                    current_instruction = None
                else:
                    if state["step"] >= len(state["challenge"]):
                        state["captured_face_frame"] = frame.copy()
                        state["awaiting_final_blink"] = True
                        state["post_capture_blink_count"] = 0
                        state["post_capture_closed_frames"] = 0
                        state["hold_started_at"] = None
                        current_instruction = None
                    else:
                        current_instruction = state["challenge"][state["step"]]

                        if current_instruction["type"] == "turn_left":
                            if direction == "Turned Left":
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds

                        elif current_instruction["type"] == "turn_right":
                            if direction == "Turned Right":
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds

                        elif current_instruction["type"] == "blink":
                            if direction == "Forward":
                                if avg_ear < blink_threshold:
                                    state["closed_frames"] += 1
                                else:
                                    if state["closed_frames"] >= min_closed_frames:
                                        state["blink_count"] += 1
                                    state["closed_frames"] = 0
                            else:
                                state["closed_frames"] = 0

                            if state["blink_count"] >= current_instruction["count"]:
                                state["step"] += 1
                                state["waiting_until"] = now + step_delay_seconds
                                state["blink_count"] = 0
                                state["closed_frames"] = 0

                        elif current_instruction["type"] == "forward_hold":
                            if direction == "Forward":
                                if state["hold_started_at"] is None:
                                    state["hold_started_at"] = now

                                held_for = now - state["hold_started_at"]
                                if held_for >= current_instruction["duration"]:
                                    state["step"] += 1
                                    state["waiting_until"] = now + step_delay_seconds
                                    state["hold_started_at"] = None
                            else:
                                state["hold_started_at"] = None

                    if state["step"] >= len(state["challenge"]) and not state["awaiting_final_blink"]:
                        state["captured_face_frame"] = frame.copy()
                        state["awaiting_final_blink"] = True
                        state["post_capture_blink_count"] = 0
                        state["post_capture_closed_frames"] = 0
                        state["hold_started_at"] = None
                        current_instruction = None
                    elif state["step"] < len(state["challenge"]):
                        if now < state["waiting_until"]:
                            current_instruction = None
                        else:
                            current_instruction = state["challenge"][state["step"]]

            elif state["awaiting_final_blink"] and not state["passed"]:
                if direction == "Forward":
                    if avg_ear < blink_threshold:
                        state["post_capture_closed_frames"] += 1
                    else:
                        if state["post_capture_closed_frames"] >= min_closed_frames:
                            state["post_capture_blink_count"] += 1
                        state["post_capture_closed_frames"] = 0
                else:
                    state["post_capture_closed_frames"] = 0

                if state["post_capture_blink_count"] >= 1:
                    state["passed"] = True
                    state["awaiting_final_blink"] = False
                    current_instruction = None

            cv2.putText(frame, f"EAR: {avg_ear:.3f}", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Direction: {direction}", (30, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 255), 2)
            cv2.putText(frame, f"Offset: {nose_offset:.3f}", (30, 120),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 255), 2)
            cv2.putText(frame, f"Step: {state['step']}/{len(state['challenge'])}", (30, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

            if state["awaiting_final_blink"] and not state["passed"]:
                instruction_text = f"Face captured. Blink once to confirm liveness ({state['post_capture_blink_count']}/1)"
                cv2.putText(frame, instruction_text, (30, 210),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

            elif current_instruction is not None and not state["passed"]:
                instruction_text = step_to_instruction(current_instruction)
                if current_instruction["type"] == "blink":
                    instruction_text += f" ({state['blink_count']}/{current_instruction['count']})"
                elif current_instruction["type"] == "forward_hold" and state["hold_started_at"] is not None:
                    held_for = now - state["hold_started_at"]
                    instruction_text += f" ({min(held_for, current_instruction['duration']):.1f}/{current_instruction['duration']:.1f}s)"

                cv2.putText(frame, instruction_text, (30, 210),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 255), 2)

            if now < state["waiting_until"] and not state["passed"] and not state["awaiting_final_blink"]:
                remaining = state["waiting_until"] - now
                cv2.putText(frame, f"Next step in {remaining:.1f}s", (30, 250),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 0), 2)

            if state["passed"]:
                cv2.putText(frame, "Liveness Passed - ready for recognition", (30, 300),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 200, 0), 2)

        cv2.imshow(window_name, frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("r"):
            state = make_new_state(step_delay_seconds)
            current_instruction = None
            retry_available_at = None
            print("New challenge:", [step_to_instruction(s) for s in state["challenge"]])

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

cap.release()
cv2.destroyAllWindows()

Selected challenge: ['Turn head right', 'Face forward and blink 2 times', 'Turn head right', 'Face forward and hold still for 2 seconds']


I0000 00:00:1776741990.219500   19395 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776741990.265875   65007 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 590.48.01), renderer: NVIDIA GeForce RTX 5070 Ti/PCIe/SSE2
W0000 00:00:1776741990.268344   64993 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776741990.283005   64994 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
